# yolo (OPTIONAL): an alternative piece detector

**Optional, not part of the M1-M5 deliverables.** Instead of classifying 64 fixed crops, a
detector (Ultralytics YOLO) finds every piece as a box + class in the warped image. Possible
advantages: it also gives the piece position (useful for `placement.ipynb`) and copes with
pieces that are off their square. Cost: box labels, a heavier model, another dependency.

Labels come for free from M2 capture: the manifest says which piece is on which square, and
the crop box of that square (`capture.ipynb: crop_box`) is a usable approximate bounding box.
Dataset config: `configs/yolo_data.yaml` at the repo root (12 classes = `CLASSES[1:]`).

**TODO** (only if you pursue this)
- [ ] tighter boxes (piece height from `board.yaml: pieces.height`, perspective lean)
- [ ] `train_yolo` on Colab (`pip install ultralytics`), compare with the ResNet-18 crops
      on the same test sessions (convert detections to 64 labels via `placement.ipynb: which_square`)

In [ ]:
from __future__ import annotations

from pathlib import Path

import chess
import numpy as np
import yaml

%run ./capture.ipynb

In [ ]:
YOLO_NAMES = CLASSES[1:]                     # 12 piece classes
YOLO_DATA_YAML = REPO_ROOT / "configs" / "yolo_data.yaml"


def label_to_yolo_class(label: str) -> int:
    if label == "empty":
        raise ValueError("empty squares have no box")
    return CLASS_TO_INDEX[label] - 1

## Boxes from a known board
YOLO label line: `class x_center y_center width height`, all normalized to [0, 1].

In [ ]:
def yolo_box(box: tuple[int, int, int, int], img_w: int, img_h: int) -> tuple[float, float, float, float]:
    """Pixel box (x0, y0, x1, y1), clipped to the image -> normalized (xc, yc, w, h)."""
    x0, y0, x1, y1 = box
    x0, x1 = np.clip([x0, x1], 0, img_w)
    y0, y1 = np.clip([y0, y1], 0, img_h)
    return ((x0 + x1) / 2 / img_w, (y0 + y1) / 2 / img_h, (x1 - x0) / img_w, (y1 - y0) / img_h)


def yolo_lines(board: chess.Board, geom: BoardGeometry, warp, crop, img_w: int, img_h: int,
               widen: float = 0.1) -> list[str]:
    """One label line per piece. Box = the square's crop box with a narrower `widen`."""
    narrow = type(crop)(**{**vars(crop), "widen": widen})
    lines = []
    for sq, lab in enumerate(board_to_labels(board)):
        if lab == "empty":
            continue
        xc, yc, w, h = yolo_box(crop_box(sq, geom, warp, narrow), img_w, img_h)
        lines.append(f"{label_to_yolo_class(lab)} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
    return lines


def write_yolo_sample(root: str | Path, split: str, name: str, warped: np.ndarray,
                      board: chess.Board, geom: BoardGeometry, warp, crop) -> None:
    """Write images/<split>/<name>.png and labels/<split>/<name>.txt under root."""
    import cv2
    root = Path(root)
    (root / "images" / split).mkdir(parents=True, exist_ok=True)
    (root / "labels" / split).mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(root / "images" / split / f"{name}.png"), warped)
    h, w = warped.shape[:2]
    (root / "labels" / split / f"{name}.txt").write_text("\n".join(yolo_lines(board, geom, warp, crop, w, h)) + "\n")

## Training (TODO)

In [ ]:
def train_yolo(data_yaml: str | Path = YOLO_DATA_YAML, model: str = "yolov8n.pt",
               epochs: int = 50, imgsz: int = 1024):
    """TODO (optional): fine-tune a YOLO model on the exported dataset.

    from ultralytics import YOLO
    YOLO(model).train(data=str(data_yaml), epochs=epochs, imgsz=imgsz)
    Decide the image size (the default warp is 1000 px), the augmentations, and how to turn
    detections into a (64, 13) array for board_assembly.ipynb (box base point ->
    placement.ipynb: which_square, class score -> probability row).
    """
    raise NotImplementedError("TODO (optional): train_yolo")

## Self-check

In [ ]:
import tempfile

_y = yaml.safe_load(YOLO_DATA_YAML.read_text())
assert [_y["names"][i] for i in range(12)] == list(YOLO_NAMES)
assert label_to_yolo_class("P") == 0 and label_to_yolo_class("k") == 11

_yc = load_config("configs/camera.yaml")
_yg = load_board_geometry()
_lines = yolo_lines(chess.Board(), _yg, _yc.warp, _yc.crop, 1000, 1000)
assert len(_lines) == 32
_vals = np.array([[float(v) for v in l.split()[1:]] for l in _lines])
assert (_vals >= 0).all() and (_vals <= 1).all()
with tempfile.TemporaryDirectory() as _t:
    write_yolo_sample(_t, "train", "000000", np.zeros((1000, 1000, 3), np.uint8), chess.Board(), _yg, _yc.warp, _yc.crop)
    assert len((Path(_t) / "labels/train/000000.txt").read_text().split("\n")) == 33
print("yolo OK")